In [2]:
import camelot
import pandas as pd

pdf_to_read = "Remittance_Cenco.pdf"

# -------------------------
# Step 1: Extraer todas las tablas excepto la última página
# -------------------------
tables = camelot.read_pdf(pdf_to_read, pages='1-25', flavor='stream', strip_text='\n')

# -------------------------
# Step 2: Concatenar todas las tablas detectadas
# -------------------------
df_all = pd.concat([table.df for table in tables], ignore_index=True)

# -------------------------
# Step 3: Detectar la fila de encabezado real
# -------------------------
header_row_idx = df_all[df_all.apply(lambda row: row.astype(str).str.contains('VOUCHER').any(), axis=1)].index[0]
df_all.columns = df_all.iloc[header_row_idx]
df_all = df_all.drop(index=list(range(header_row_idx + 1))).reset_index(drop=True)
df_all = df_all[~df_all.apply(lambda row: all(row.astype(str) == df_all.columns.astype(str)), axis=1)].reset_index(drop=True)

# -------------------------
# Ahora sí podemos usar df_all.columns como referencia
ref_columns = df_all.columns.tolist()

# -------------------------
# Step 4: Extraer página 26 con Camelot lattice
# -------------------------
tables_last = camelot.read_pdf(pdf_to_read, pages='26', flavor='lattice', strip_text='\n')

df_last_records = []

for table in tables_last:
    df = table.df
    for _, row in df.iterrows():
        row_list = row.tolist()
        # Ajustar cantidad de columnas
        if len(row_list) < len(ref_columns):
            row_list += [''] * (len(ref_columns) - len(row_list))
        elif len(row_list) > len(ref_columns):
            row_list = row_list[:len(ref_columns)]
        df_last_records.append(row_list)

if df_last_records:
    df_last = pd.DataFrame(df_last_records, columns=ref_columns)
    print("✅ Página 26 extraída con Camelot. Registros:", len(df_last))
    df_all = pd.concat([df_all, df_last], ignore_index=True)
else:
    print("⚠️ No se detectaron registros en página 26")


/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.2204959568734)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.2112062663186)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.215777188329)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables fou

⚠️ No se detectaron registros en página 26
